In [ ]:
from google.colab import files
uploaded = files.upload()

Saving DataCoSupplyChainDataset.csv to DataCoSupplyChainDataset.csv


Google Colab runs on Google's servers, not your computer. So your local file doesn't exist there automatically — you have to manually upload it first. Think of it like attaching a file before sending an email.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('DataCoSupplyChainDataset.csv', encoding='latin-1')

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 180519
Columns: 53


This file was not saved in standard UTF-8 format. It has some special characters (like accented letters in city names — São Paulo, etc.). If you load without this, Python throws a UnicodeDecodeError. latin-1 tells Python to read those special characters correctly. Always use this for DataCo.

In [ ]:
# See first 5 rows
df.head()

,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,...,product_category_id,product_name,product_price,product_status,shipping_date_dateorders,shipping_mode,shipping_delay_days,order_month,order_year,order_month_year
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,73,Smart watch,327.75,0,2018-02-03 22:56:00,Standard Class,-1,1,2018,2018-01
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,73,Smart watch,327.75,0,2018-01-18 12:27:00,Standard Class,1,1,2018,2018-01
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,73,Smart watch,327.75,0,2018-01-17 12:06:00,Standard Class,0,1,2018,2018-01
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,73,Smart watch,327.75,0,2018-01-16 11:45:00,Standard Class,-1,1,2018,2018-01
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,73,Smart watch,327.75,0,2018-01-15 11:24:00,Standard Class,-2,1,2018,2018-01


Order date (DateOrders) looks like a date but it's stored as text — needs fixing later.

Customer Password column is there — clearly useless for analysis.

Product Description column appears completely empty.

Some columns have very long messy names with brackets like Days for shipping (real).

In [ ]:
# See all column names and their data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 48 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   type                         180519 non-null  object        
 1   days_for_shipping_real       180519 non-null  int64         
 2   days_for_shipment_scheduled  180519 non-null  int64         
 3   benefit_per_order            180519 non-null  float64       
 4   sales_per_customer           180519 non-null  float64       
 5   delivery_status              180519 non-null  object        
 6   late_delivery_risk           180519 non-null  int64         
 7   category_id                  180519 non-null  int64         
 8   category_name                180519 non-null  object        
 9   customer_city                180519 non-null  object        
 10  customer_country             180519 non-null  object        
 11  customer_id               

object = text/string column. Dates stored as object are a problem — they look like dates but Python treats them as plain text, so you can't do any date math on them.

int64 = whole numbers (like 2, 3, 5 days)

float64 = decimal numbers (like profit 45.32)

0 non-null for Product Description = the entire column is empty — 1,80,519 missing values. Pure dead weight.

In [ ]:
# See missing values in each column
df.isnull().sum()

,0
type,0
days_for_shipping_real,0
days_for_shipment_scheduled,0
benefit_per_order,0
sales_per_customer,0
delivery_status,0
late_delivery_risk,0
category_id,0
category_name,0
customer_city,0


Product Description — 1,80,519 missing out of 1,80,519 rows = 100% empty. This column exists but has no data at all. Must drop.

Order Zipcode — 1,55,679 missing out of 1,80,519 = 86% missing. Almost entirely empty, useless for analysis. Must drop.

Customer Lname — only 8 missing, very small. We'll fill with 'Unknown'.

Customer Zipcode — only 3 missing, similarly tiny.

In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
cols_to_drop = [
    'Customer Email',
    'Customer Password',
    'Customer Fname',
    'Customer Lname',
    'Customer Street',
    'Customer Zipcode',
    'Order Zipcode',
    'Product Description',
    'Product Image'
]

df = df.drop(columns=cols_to_drop, errors='ignore')

print("Columns remaining:", df.shape[1])

Columns remaining: 48


Customer Email & Customer Password — personal data, zero analytical value. No analysis needs this.

Customer Fname & Customer Lname — individual names don't help in supply chain analysis. We care about segments, regions, and markets — not who "John Smith" is.

Customer Street — too granular. We already have City, State, Country for location analysis.

Customer Zipcode & Order Zipcode — 86%+ missing, and zip codes aren't useful for this project.

Product Description — 100% empty, confirmed dead column.

Product Image — contains URL links to product images. Useless for data analysis.

In [ ]:
# Remove spaces, brackets, make lowercase with underscores
df.columns = (df.columns
              .str.strip()
              .str.lower()
              .str.replace(' ', '_')
              .str.replace('(', '')
              .str.replace(')', '')
              .str.replace('__', '_'))

print(df.columns.tolist())

['type', 'days_for_shipping_real', 'days_for_shipment_scheduled', 'benefit_per_order', 'sales_per_customer', 'delivery_status', 'late_delivery_risk', 'category_id', 'category_name', 'customer_city', 'customer_country', 'customer_id', 'customer_segment', 'customer_state', 'department_id', 'department_name', 'latitude', 'longitude', 'market', 'order_city', 'order_country', 'order_customer_id', 'order_date_dateorders', 'order_id', 'order_item_cardprod_id', 'order_item_discount', 'order_item_discount_rate', 'order_item_id', 'order_item_product_price', 'order_item_profit_ratio', 'order_item_quantity', 'sales', 'order_item_total', 'order_profit_per_order', 'order_region', 'order_state', 'order_status', 'product_card_id', 'product_category_id', 'product_name', 'product_price', 'product_status', 'shipping_date_dateorders', 'shipping_mode', 'shipping_delay_days', 'order_month', 'order_year', 'order_month_year']


Later when import this cleaned file into SQL, column names with spaces and brackets cause errors or require annoying backtick quoting everywhere. Clean snake_case names like order_date_dateorders work directly in SQL, Python, and Power BI without any special treatment. This is standard professional practice.

In [ ]:
df['order_date_dateorders'] = pd.to_datetime(
    df['order_date_dateorders'],
    format='mixed',
    dayfirst=False
)

df['shipping_date_dateorders'] = pd.to_datetime(
    df['shipping_date_dateorders'],
    format='mixed',
    dayfirst=False
)

print(df[['order_date_dateorders','shipping_date_dateorders']].dtypes)

order_date_dateorders       datetime64[ns]
shipping_date_dateorders    datetime64[ns]
dtype: object


Before this step, the date columns were just text — Python had no idea "1/31/2018" was a date. Now they're datetime64 objects, which means Python understands them as real dates. This unlocks powerful operations:

Extract month → df['order_date_dateorders'].dt.month
Extract year → df['order_date_dateorders'].dt.year
Calculate difference between two dates → actual shipping delay analysis
Group orders by week or month → demand forecasting

Without this fix, none of the time-series analysis in later steps would work.

In [ ]:
# Confirm no missing values remain
print("Total nulls:", df.isnull().sum().sum())

Total nulls: 0


After dropping the 9 useless columns in Step 4 (especially Product Description which was 100% empty and Order Zipcode which was 86% empty), all missing values got automatically eliminated. The remaining columns like Customer City, Order Region, Shipping Mode etc. are fully complete with zero missing values across all 1,80,519 rows. No further action like filling or deleting rows was needed.

In [ ]:
# Shipping delay (actual - scheduled)
df['shipping_delay_days'] = df['days_for_shipping_real'] - df['days_for_shipment_scheduled']

# Extract month and year from order date
df['order_month'] = df['order_date_dateorders'].dt.month
df['order_year'] = df['order_date_dateorders'].dt.year
df['order_month_year'] = df['order_date_dateorders'].dt.to_period('M')

print(df[['shipping_delay_days','order_month','order_year']].head())

   shipping_delay_days  order_month  order_year
0                   -1            1        2018
1                    1            1        2018
2                    0            1        2018
3                   -1            1        2018
4                   -2            1        2018


shipping_delay_days — positive number = arrived late by that many days. Negative = arrived early. This is your most important column for bottleneck analysis later. You're not just counting late deliveries — you're measuring how late.

order_month and order_year — these are the building blocks for demand forecasting. When you group by month later and plot order volumes, you'll see seasonality patterns — which months have peak demand, which are slow.
order_month_year like 2018-01 — useful for time-series charts in Power BI.

These columns don't exist in the original data — you created them. This is called feature engineering, and it's a key data analyst skill.


In [ ]:
# Final shape check
print("Final shape:", df.shape)
print("Any nulls left:", df.isnull().sum().sum())
print("Dtypes summary:")
print(df.dtypes.value_counts())

# Save clean file
df.to_csv('dataco_cleaned.csv', index=False)
print("✅ Clean file saved!")

Final shape: (180519, 48)
Any nulls left: 0
Dtypes summary:
object            16
int64             15
float64           12
datetime64[ns]     2
int32              2
period[M]          1
Name: count, dtype: int64
✅ Clean file saved!


Started with 53 columns, now I have 48 (44 original remaining + 4 new ones created)

Zero nulls remaining — clean and complete.

2 datetime columns — dates are properly formatted.

The file dataco_cleaned.csv is now saved in Colab.

In [ ]:
from google.colab import files
files.download('dataco_cleaned.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# What markets are in the data?
print("=== MARKETS ===")
print(df['market'].value_counts())

print("\n=== DELIVERY STATUS ===")
print(df['delivery_status'].value_counts())

print("\n=== SHIPPING MODES ===")
print(df['shipping_mode'].value_counts())

print("\n=== LATE DELIVERY RATE ===")
late_pct = df['late_delivery_risk'].mean() * 100
print(f"{late_pct:.1f}% of orders have late delivery risk")

=== MARKETS ===
market
LATAM           51594
Europe          50252
Pacific Asia    41260
USCA            25799
Africa          11614
Name: count, dtype: int64

=== DELIVERY STATUS ===
delivery_status
Late delivery        98977
Advance shipping     41592
Shipping on time     32196
Shipping canceled     7754
Name: count, dtype: int64

=== SHIPPING MODES ===
shipping_mode
Standard Class    107752
Second Class       35216
First Class        27814
Same Day            9737
Name: count, dtype: int64

=== LATE DELIVERY RATE ===
54.8% of orders have late delivery risk


54.9% late delivery rate — more than half of all orders are at risk of being late. This is a massive supply chain problem and your entire project revolves around finding WHY.

Europe is the biggest market — but is it also the most delayed? It'll find out in the SQL phase.

Standard Class is most used but is it the most delayed shipping mode? That's a key question to answer.

99,090 late deliveries out of 1,80,519 — that means almost every second order is late. A company would desperately need your analysis.